# 26 - Source multi-query selection worker 2

This is the source-only multi-query experiment, intentionally run before model aggregation.
At each live decision observation, independent source-checkpoint queries produce clean action
chunks. Each candidate receives K=5 uncertainty measurement at Euler steps `(3,4)`, and the
lowest-uncertainty clean chunk is executed. **No refinement is applied.** `N_QUERIES` defaults to
2; later values such as 3 or 4 produce separate resumable rollout IDs in the same experiment.

This is shard 2 of 4. All workers together collect 1,300 identities (10 episodes per
task), one rollout per identity. The progress table's historical column is the exact old
uncertainty-only source baseline, not an offline threshold policy. Worker 0 may first set
`EPISODE_LIMIT=1`; restore it to `None` afterward and the smoke row will be reused.

## 1. Setup a fresh GPU runtime

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration and resumable collection

In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.config import PI05_REPO_ID
from pnp.diversity import (SOURCE_MULTI_QUERY_EXPERIMENT,
    load_bootstrap_manifest, run_source_multi_query_worker)

drive.mount("/content/drive")

EPISODES_PER_TASK = 10
SHARD_COUNT = 4
SHARD_INDEX = 2
EPISODE_LIMIT = None  # worker-0 smoke: set 1 once, then restore None
N_QUERIES = 2
EXPERIMENT = SOURCE_MULTI_QUERY_EXPERIMENT
MANIFEST_PATH = Path(
    "/content/drive/MyDrive/pnp_diversity_v2/bootstrap_manifest_finetuned_v2.json")
manifest = load_bootstrap_manifest(MANIFEST_PATH)
assert manifest["source_model"] == PI05_REPO_ID, manifest["source_model"]
SOURCE_MODEL_REVISION = manifest["source_model_revision"]
assert SOURCE_MODEL_REVISION, "v2 manifest is missing source_model_revision"

print({"experiment": EXPERIMENT, "queries": N_QUERIES, "refinement": False,
       "episodes_per_task": EPISODES_PER_TASK,
       "shard_count": SHARD_COUNT, "shard_index": SHARD_INDEX,
       "episode_limit": EPISODE_LIMIT,
       "manifest_hash": manifest["manifest_hash"],
       "source_model_revision": SOURCE_MODEL_REVISION})
run_source_multi_query_worker(
    episodes_per_task=EPISODES_PER_TASK, episode_limit=EPISODE_LIMIT,
    num_queries=N_QUERIES,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    manifest_hash=manifest["manifest_hash"],
    source_model_revision=SOURCE_MODEL_REVISION, experiment=EXPERIMENT)